# One-Hot Encoding (OHE)
### Advanced Mechanics, Multi-Collinearity Prevention, and Production Pipelines

---

### Scope & Focus of This Notebook
- **Primary Topic**: One-Hot Encoding (OHE) deep dive for nominal categorical features.
- **What You Will Learn**:
  1. Foundational principle: Creating binary indicator columns where categories are equidistant in Euclidean space.
  2. Manual Python dummy creation before calling library APIs.
  3. Rapid EDA prototyping with Pandas `pd.get_dummies()`.
  4. The Dummy Variable Trap: Proving multicollinearity ($r = -1.0$) and resolving it with `drop='first'`.
  5. Scikit-Learn `OneHotEncoder`: Production-grade workflows, `fit()` vs. `transform()`, and preventing data leakage.
  6. Unseen category handling: Preventing production pipeline crashes using `handle_unknown="ignore"`.
  7. The modern production standard: `drop='if_binary'`.
  8. High cardinality mitigation: Pruning rare tail categories using `min_frequency`.
  9. Memory optimization: Dense arrays vs. Scipy sparse matrices (`sparse_output`).
- **What We Will NOT Cover Here**:
  - Foundational Ordinal vs. Nominal classification was covered in **`03_Encoding_Categorical_Data`**.
  - Multi-column preprocessing coordination belongs to **`05_Column_Transformer`**.
  - Complete end-to-end model workflows belong to **`06_Machine_Learning_Pipeline`**.

---

## 1. Setup & Environment Verification

### What are we doing?
We import data manipulation, encoding, and visualization libraries.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split

# Visual styling
sns.set_theme(style="whitegrid")
plt.rcParams['font.family'] = 'DejaVu Sans'

print("Libraries imported successfully.")


Libraries imported successfully.


## 2. Why Ordinary Integer Encoding Fails for Nominal Data

Suppose we assign arbitrary integers to `City`:
- $\text{Delhi} \to 0$
- $\text{Mumbai} \to 1$
- $\text{Kolkata} \to 2$

Mathematical distances:
- $\text{dist}(\text{Delhi}, \text{Mumbai}) = |0 - 1| = 1$
- $\text{dist}(\text{Delhi}, \text{Kolkata}) = |0 - 2| = 2$

The model assumes Kolkata is twice as far from Delhi as Mumbai is, and that Kolkata is greater than Delhi. This introduces spurious linear relationships into linear models, neural networks, and distance-based algorithms.


## 3. Manual Understanding: Deriving One-Hot Encoding from Scratch

### What are we doing?
We manually convert a categorical column into binary indicator columns using pure Python logic.

#### Why are we doing this?
To understand how each unique category maps to an orthogonal axis in vector space.

#### The Equidistance Property:
In one-hot vector space:
- $\text{Delhi} = [1, 0, 0]$
- $\text{Mumbai} = [0, 1, 0]$
- $\text{Kolkata} = [0, 0, 1]$

The Euclidean distance between ANY pair of categories is strictly:
$$\sqrt{(1 - 0)^2 + (0 - 1)^2 + (0 - 0)^2} = \sqrt{1 + 1} = \sqrt{2} \approx 1.414$$
**Every category is mathematically equidistant from every other category!**


In [2]:
cities = ['Delhi', 'Mumbai', 'Kolkata', 'Delhi', 'Mumbai']
unique_cities = sorted(list(set(cities)))

# Manual one-hot matrix generation
manual_matrix = []
for city in cities:
    row = [1 if city == u else 0 for u in unique_cities]
    manual_matrix.append(row)

manual_ohe_df = pd.DataFrame(manual_matrix, columns=[f"City_{c}" for c in unique_cities])
manual_ohe_df.insert(0, 'Original_City', cities)
manual_ohe_df


,Original_City,City_Delhi,City_Kolkata,City_Mumbai
0,Delhi,1,0,0
1,Mumbai,0,0,1
2,Kolkata,0,1,0
3,Delhi,1,0,0
4,Mumbai,0,0,1


## 4. One-Hot Encoding Using Pandas: `pd.get_dummies()`

### What are we doing?
We encode nominal features using Pandas' built-in `pd.get_dummies()` function.

#### What does `pd.get_dummies()` do?
Convenience method that converts categorical columns of a DataFrame into dummy indicator variables.

#### When should you use it?
Ideal for rapid Exploratory Data Analysis (EDA) and quick prototyping.

#### Why is it NOT recommended for production ML pipelines?
`pd.get_dummies()` does not maintain persistent state:
1. It cannot remember column order across train and test sets.
2. If the test set has a category not present in the training set (or vice versa), the train and test DataFrames will end up with different numbers of columns, crashing your model.
Use Scikit-Learn's `OneHotEncoder` for production workflows.


In [3]:
toy_df = pd.DataFrame({
    'City': ['Delhi', 'Mumbai', 'Kolkata', 'Delhi'],
    'Gender': ['Male', 'Female', 'Female', 'Male']
})

# Rapid encoding for EDA
pd.get_dummies(toy_df, dtype=int)


,City_Delhi,City_Kolkata,City_Mumbai,Gender_Female,Gender_Male
0,1,0,0,0,1
1,0,0,1,1,0
2,0,1,0,1,0
3,1,0,0,0,1


## 5. The Dummy Variable Trap & Multicollinearity

### What are we doing?
We demonstrate the mathematical trap that occurs when keeping all $N$ dummy columns for an $N$-level feature.

#### The Mathematical Problem: Perfect Multicollinearity
For any observation:
$$\text{Gender\_Female} + \text{Gender\_Male} = 1 \implies \text{Gender\_Male} = 1 - \text{Gender\_Female}$$

One feature can be predicted perfectly from the other ($r = -1.0$).
In Linear and Logistic Regression, this creates a **singular matrix** ($X^T X$ is non-invertible), causing numerical instability and exploding coefficient variances.

#### The Solution: Drop One Category ($N - 1$ Encoding)
If `Gender_Female == 0`, the passenger is guaranteed to be male. Dropping one column retains 100% of the information without multicollinearity.


In [4]:
# Notice drop_first=True drops the first category (Female), keeping only Male
pd.get_dummies(toy_df[['Gender']], drop_first=True, dtype=int)


,Gender_Male
0,1
1,0
2,0
3,1


## 6. One-Hot Encoding with Scikit-Learn: `OneHotEncoder`

### What are we doing?
We apply Scikit-Learn's `OneHotEncoder` to encode categorical features.

#### What does `OneHotEncoder` do?
Production-grade estimator that learns category levels during `.fit()` and applies consistent transformations during `.transform()`.

#### Important Parameters:
- `sparse_output=False`: Returns dense NumPy arrays instead of Scipy sparse matrices.
- `drop='first'`: Automatically drops the first category to avoid multicollinearity.
- `handle_unknown='ignore'`: Crucial parameter for production inference.


In [5]:
ohe = OneHotEncoder(sparse_output=False, drop='first')
encoded_arr = ohe.fit_transform(toy_df[['City', 'Gender']])

encoded_df = pd.DataFrame(encoded_arr, columns=ohe.get_feature_names_out(['City', 'Gender']))
encoded_df


,City_Kolkata,City_Mumbai,Gender_Male
0,0.0,0.0,1.0
1,0.0,1.0,0.0
2,1.0,0.0,0.0
3,0.0,0.0,1.0


## 7. Train / Test Workflow & Preventing Data Leakage

### What are we doing?
We demonstrate the strict split-first protocol:
1. `train_test_split` creates `X_train` and `X_test`.
2. `ohe.fit_transform(X_train)` learns categories from `X_train` only.
3. `ohe.transform(X_test)` transforms `X_test` using the learned categories.


In [6]:
df_sample = pd.DataFrame({
    'City': ['Delhi', 'Mumbai', 'Kolkata', 'Delhi', 'Mumbai', 'Delhi'],
    'Gender': ['Male', 'Female', 'Male', 'Female', 'Male', 'Female'],
    'Target': [1, 0, 1, 1, 0, 0]
})

X = df_sample[['City', 'Gender']]
y = df_sample['Target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

encoder = OneHotEncoder(sparse_output=False, drop='first')
X_train_encoded = encoder.fit_transform(X_train)
X_test_encoded = encoder.transform(X_test)

print("Learned Categories per Feature:", encoder.categories_)
print(f"X_train_encoded shape: {X_train_encoded.shape}")
print(f"X_test_encoded shape:  {X_test_encoded.shape}")


Learned Categories per Feature: [array(['Delhi', 'Kolkata', 'Mumbai'], dtype=object), array(['Female', 'Male'], dtype=object)]
X_train_encoded shape: (4, 3)
X_test_encoded shape:  (2, 3)


## 8. Handling Unseen Categories with `handle_unknown="ignore"`

### What are we doing?
In production, a user might submit a city that never appeared in the training set (e.g. `'Bengaluru'`).

#### The Problem:
By default, Scikit-Learn raises a `ValueError: Found unknown categories ['Bengaluru'] in column 0 during transform`.
In a production web application, this causes a 500 internal server error and crashes the service!

#### The Solution:
Setting `handle_unknown="ignore"` instructs the encoder to silently output a vector of all **0s** for that feature without throwing an error.


In [7]:
# Production-safe encoder
robust_ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
robust_ohe.fit(pd.DataFrame({'City': ['Delhi', 'Mumbai']}))

# Production test data with an unseen category ('Bengaluru')
unseen_data = pd.DataFrame({'City': ['Delhi', 'Bengaluru', 'Mumbai']})
encoded_unseen = robust_ohe.transform(unseen_data)

pd.DataFrame(encoded_unseen, columns=robust_ohe.get_feature_names_out(['City']), index=['Delhi', 'Bengaluru (Unseen)', 'Mumbai'])


,City_Delhi,City_Mumbai
Delhi,1.0,0.0
Bengaluru (Unseen),0.0,0.0
Mumbai,0.0,1.0


## 9. The Modern Production Standard: `drop='if_binary'`

### Why is `drop='if_binary'` the best practice?
- For binary features (`Gender`: Male, Female), keeping 2 columns causes multicollinearity $\implies$ drops one column ($1$ column output).
- For multi-category features (`City`: Delhi, Mumbai, Kolkata), keeping all 3 columns allows `handle_unknown="ignore"` to safely fall back to an all-zeros vector without ambiguity.
Scikit-Learn natively supports `drop='if_binary'`!


In [8]:
smart_ohe = OneHotEncoder(sparse_output=False, drop='if_binary', handle_unknown='ignore')
smart_encoded = smart_ohe.fit_transform(toy_df)

pd.DataFrame(smart_encoded, columns=smart_ohe.get_feature_names_out(['City', 'Gender']))


,City_Delhi,City_Kolkata,City_Mumbai,Gender_Male
0,1.0,0.0,0.0,1.0
1,0.0,0.0,1.0,0.0
2,0.0,1.0,0.0,0.0
3,1.0,0.0,0.0,1.0


## 10. High Cardinality & Rare Category Grouping (`min_frequency`)

### The Curse of Dimensionality:
If a dataset has 500 postal ZIP codes or 200 car brands, OHE expands the dataset by hundreds of sparse columns, exhausting memory and increasing overfitting risk.

### The Solution: `min_frequency`
Scikit-Learn provides `min_frequency` to automatically group categories that appear below a threshold into a single composite category called `"infrequent_sklearn"`.


In [9]:
car_brands = pd.DataFrame({
    'Brand': ['Toyota', 'Toyota', 'Toyota', 'Ford', 'Ford', 'Honda', 'Ferrari', 'Bugatti', 'Pagani']
})

freq_ohe = OneHotEncoder(sparse_output=False, min_frequency=2)
freq_encoded = freq_ohe.fit_transform(car_brands)

pd.DataFrame(freq_encoded, columns=freq_ohe.get_feature_names_out(['Brand']))


,Brand_Ford,Brand_Toyota,Brand_infrequent_sklearn
0,0.0,1.0,0.0
1,0.0,1.0,0.0
2,0.0,1.0,0.0
3,1.0,0.0,0.0
4,1.0,0.0,0.0
5,0.0,0.0,1.0
6,0.0,0.0,1.0
7,0.0,0.0,1.0
8,0.0,0.0,1.0


---

## What We Learned
- **Equidistance Preservation**: OHE maps categories to orthogonal binary axes, preserving equal pairwise Euclidean distances ($\sqrt{2} \approx 1.414$).
- **The Dummy Variable Trap**: Keeping all $N$ levels of a categorical feature causes perfect multicollinearity ($r = -1.0$), crashing linear model inversions.
- **Scikit-Learn `OneHotEncoder`**: Production standard that preserves feature schema, supports stateful fit/transform, and integrates into pipelines.
- **Crash Prevention via `handle_unknown="ignore"`**: Silently encodes unseen inference categories into zero-vectors rather than crashing production.
- **The `drop='if_binary'` Standard**: Drops redundant columns for 2-class binary features while preserving all levels for multi-class features.
- **Dimensionality Control**: `min_frequency` collapses long-tail rare categories into a unified infrequent bucket.

---

## Important Takeaways
1. **Never use `pd.get_dummies()` in production inference pipelines**: It cannot guarantee identical column count and order across unseen test samples. Use `OneHotEncoder`.
2. **Always evaluate cardinality before applying OHE**: For features with $> 50$ distinct categories, apply `min_frequency` or alternative encodings (Target Encoding, Frequency Encoding) to avoid dimensionality explosion.
3. **Use `sparse_output=True` on wide datasets**: When encoding high-cardinality features, sparse matrices save gigabytes of memory.

---

## Common Mistakes
1. **Fitting OHE on the entire dataset before splitting**: Leaks test category presence into the encoder.
2. **Using `drop='first'` simultaneously with `handle_unknown='ignore'`**: When the first category is dropped, an all-zero vector represents the dropped category. If an unseen category also becomes all zeros, the model cannot distinguish between them.
3. **Leaving rare single-instance categories unhandled**: Categories with 1 observation create uninformative, overfitting indicator columns.

---

## Final Mental Model

```text
┌────────────────────────────────────────────────────────────────────────┐
│                        ONE-HOT ENCODING STRATEGY                       │
└────────────────────────────────────────────────────────────────────────┘
                               │
                Is cardinality low or high?
                               │
            ┌──────────────────┴──────────────────┐
            ▼                                     ▼
     Low Cardinality                       High Cardinality
     (< 10-15 levels)                      (> 20-50 levels)
            │                                     │
            ▼                                     ▼
   OneHotEncoder()                        Use min_frequency=k
   drop='if_binary'                       or Frequency / Target Encoding
   handle_unknown='ignore'
```
